# Embedding APIs and Model Selection

Production RAG and semantic search rarely train embedding models from scratch. You call an **Embeddings API**: send text, receive a vector. Providers host the model, handle scaling, and ship improvements — your job is to **choose the right model**, **batch efficiently**, **respect token limits**, and **keep index and query settings consistent**.

This notebook covers the OpenAI Embeddings API in depth (primary demos), request/response anatomy, batching and token accounting, the `dimensions` parameter, cost estimation, a structured **model selection** framework, and comparisons with Gemini and local open-source options.

Prerequisites: **01–02** in this section and **03. LLM API Integration**.

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

---

## 1. Embeddings API vs Chat API

LLM providers expose **separate endpoints** for generation and embeddings:

| API | Input | Output | Use |
|-----|-------|--------|-----|
| **Chat completions** | Messages (roles) | Generated text | Q&A, agents, summarization |
| **Embeddings** | String(s) | Float vectors | Search, RAG retrieval, clustering |

OpenAI example:

```python
# Chat — NOT for retrieval vectors
client.chat.completions.create(model="gpt-4o-mini", messages=[...])

# Embeddings — retrieval vectors
client.embeddings.create(model="text-embedding-3-small", input=["chunk text"])
```

Do not ask a chat model to "output an embedding" in JSON — use the dedicated endpoint for consistent, optimized vectors.

---

## 2. OpenAI Embedding Model Families

Check [OpenAI models documentation](https://platform.openai.com/docs/models) for current IDs and limits.

| Model | Default dims | Typical use | Trade-off |
|-------|--------------|-------------|----------|
| `text-embedding-3-small` | 1536 | **Default** — learning, high-volume RAG | Best cost/quality balance |
| `text-embedding-3-large` | 3072 | Hard retrieval, domain with subtle distinctions | Higher cost & storage |
| `text-embedding-ada-002` | 1536 | Legacy indexes | Migrate when practical |

### 2.1 What Differs Between Models

- **Training data and objectives** — affect what "similar" means
- **Vector dimensionality** — storage and search cost
- **Price per token** — ingest cost at scale
- **Quality on benchmarks** — MTEB and your own eval set (notebook 09) matter more than marketing

### 2.2 One Model Per Index

A vector index is tied to a **specific embedding model** (and `dimensions` setting). Mixing models breaks geometry — always **re-embed the full corpus** when switching.

---

## 3. Request and Response Anatomy

### 3.1 Request

```python
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=["text A", "text B"],      # str or list[str]
    dimensions=512,                     # optional, v3 models only
    encoding_format="float",            # default
)
```

| Field | Notes |
|-------|-------|
| `model` | Embedding model ID |
| `input` | Single string or list of strings |
| `dimensions` | Optional reduction (v3); same at index and query time |

### 3.2 Response

```python
response.data[i].embedding   # list[float]
response.data[i].index         # position in input batch
response.usage.total_tokens    # billed tokens
```

Results may not be sorted by `index` — always match via `item.index` when batching.

---

## 4. Batching and Throughput

### 4.1 Why Batch

| Single-string loop | Batched request |
|--------------------|-----------------|
| N HTTP round trips | 1 round trip |
| Higher latency | Lower amortized latency |
| More connection overhead | Better for ingest jobs |

### 4.2 Batching Guidelines

- Batch **as many chunks per request** as limits allow (often hundreds of strings)
- Keep batch sizes stable in ingest pipelines for monitoring
- On **429 rate limit**, exponential backoff and smaller batches
- Preserve **id → vector** mapping with explicit indices or parallel lists

### 4.3 Token Limits

Each input string has a **maximum token count** per model (see docs). Texts exceeding the limit must be **truncated or chunked** (notebook 04) — never silently assume the API will embed a full book.

---

## 5. Token Estimation with tiktoken

Embeddings are billed by **input tokens**. Use `tiktoken` to estimate ingest cost before embedding large corpora (same library as chat token counting in **03. LLM API Integration**).

In [ ]:
try:
    import tiktoken

    enc = tiktoken.get_encoding("cl100k_base")
    samples = [
        "Short chunk.",
        "Alembic manages database schema migrations for SQLAlchemy projects.",
        " " * 500 + "padding words " * 50,
    ]
    for s in samples:
        n = len(enc.encode(s))
        print(f"{n:4d} tokens  |  {s[:60].strip()}...")
except ImportError:
    print("Install tiktoken: pip install tiktoken")

---

## 6. The `dimensions` Parameter

OpenAI `text-embedding-3-small` and `text-embedding-3-large` support **Matryoshka-style** shorter vectors via `dimensions`:

$$\mathbf{e} \in \mathbb{R}^{d} \quad \text{where } d \leq d_{\max}$$

| Setting | Effect |
|---------|--------|
| Default (e.g. 1536) | Full model capacity |
| Reduced (e.g. 256) | Less storage, faster search, small quality loss |

**Critical:** index and query must use the **same** `model` and `dimensions`. Changing either requires a **full re-index**.

---

## 7. Model Selection Framework

Answer these in order:

1. **Deployment** — Cloud API, VPC, or fully offline?
2. **Privacy** — Can text leave your network?
3. **Scale** — How many chunks and queries per day?
4. **Quality bar** — Labeled retrieval eval (notebook 09)
5. **Existing stack** — Already on OpenAI, Gemini, or Postgres?

| Profile | Starting choice |
|---------|-----------------|
| Learning / default RAG | `text-embedding-3-small` |
| Cost-sensitive at huge scale | `3-small` + lower `dimensions` |
| Retrieval eval below target | Try `3-large` or tune chunking first |
| Air-gapped / no API | `sentence-transformers` locally |
| Google-centric stack | Gemini embedding API |
| Enterprise rerank + embed | Evaluate Cohere / Voyage on your data |

**Process:** baseline with `3-small` → measure Recall@k → upgrade model or chunking only if eval proves the need.

---

## 8. Provider Comparison

### 8.1 OpenAI (Primary in This Curriculum)

```python
from openai import OpenAI
client = OpenAI()
client.embeddings.create(model="text-embedding-3-small", input=["..."])
```

### 8.2 Google Gemini

```python
from google import genai
client = genai.Client(api_key=GOOGLE_API_KEY)
client.models.embed_content(model="text-embedding-004", contents="...")
```

Use when the rest of your stack is Gemini; do not mix Gemini embeddings with OpenAI index without full re-embed.

### 8.3 Local: sentence-transformers

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
vectors = model.encode(["chunk one", "chunk two"])
```

Pros: no per-token API cost, data stays local. Cons: you manage GPU/CPU, model updates, and ops.

### 8.4 Specialty Providers

**Cohere**, **Voyage**, and others target retrieval-heavy workloads. Worth A/B testing when OpenAI baselines underperform on **your** eval set — not as a default starting point.

---

## 9. Caching and Idempotency

| Pattern | When |
|---------|------|
| **Cache query embeddings** | Same search repeated (popular FAQs) |
| **Cache chunk embeddings on disk** | Rebuild index without re-calling API if source unchanged |
| **Content-hash keys** | `sha256(text) → vector` dedupes identical chunks |

Invalidate cache when `model` or `dimensions` changes.

---

## 10. Cost Estimation

$$\text{ingest cost} \approx \sum_{\text{chunks}} \text{tokens}(\text{chunk}) \times \text{price}_{\text{embed}}$$

$$\text{query cost per day} \approx \text{queries/day} \times \text{avg query tokens} \times \text{price}_{\text{embed}}$$

**Example sketch** (plug in current pricing):

- 10,000 chunks × 120 tokens each = 1.2M ingest tokens (one-time per re-index)
- 1,000 searches/day × 15 tokens = 15k query tokens/day

Ingest dominates when you re-index often; queries dominate at high traffic with stable indexes. **Chunking** directly multiplies ingest tokens.

---

## 11. Demonstration Setup

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL_SMALL = "text-embedding-3-small"


def cosine(a: list[float], b: list[float]) -> float:
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))


def embed(model: str, texts: list[str], dimensions: int | None = None) -> tuple[list[list[float]], int]:
    kwargs = {"model": model, "input": texts}
    if dimensions is not None:
        kwargs["dimensions"] = dimensions
    r = client.embeddings.create(**kwargs)
    ordered = sorted(r.data, key=lambda x: x.index)
    return [d.embedding for d in ordered], r.usage.total_tokens


print("Ready.")

---

## 12. Demonstration: Single and Batch Requests

In [ ]:
single_text = "JWT bearer tokens authenticate REST API requests."
vecs_one, tok_one = embed(MODEL_SMALL, [single_text])

batch_texts = [
    "FastAPI dependency injection pattern.",
    "SQLAlchemy ORM session lifecycle.",
    "Pytest fixtures for integration tests.",
]
vecs_batch, tok_batch = embed(MODEL_SMALL, batch_texts)

print("Single input:")
print(f"  dims={len(vecs_one[0])}  tokens={tok_one}")
print("\nBatch input:")
print(f"  count={len(vecs_batch)}  tokens={tok_batch}")
for i, (text, vec) in enumerate(zip(batch_texts, vecs_batch)):
    print(f"  [{i}] dim={len(vec)}  {text}")

---

## 13. Demonstration: Full vs Reduced Dimensions

Same text at 1536 vs 256 dimensions — vectors differ in length but should remain semantically useful.

In [ ]:
text = "Database migration scripts managed by Alembic for SQLAlchemy."
query = "How do I apply schema migrations?"

for dim in (None, 256):
    doc_vec, _ = embed(MODEL_SMALL, [text], dimensions=dim)
    q_vec, _ = embed(MODEL_SMALL, [query], dimensions=dim)
    label = "default" if dim is None else str(dim)
    print(f"dimensions={label:7s}  len={len(doc_vec[0]):4d}  cosine(doc,query)={cosine(doc_vec[0], q_vec[0]):.4f}")

Compare reduced dimensions on notebook **09** eval before committing production indexes to 256-d.

---

## 14. Demonstration: `small` vs `large` Model

Optional comparison — two API calls, two models. Rankings usually agree; scores differ.

In [ ]:
MODEL_LARGE = "text-embedding-3-large"
corpus = [
    "Alembic revision files track SQLAlchemy schema changes.",
    "JWT access tokens expire after a configured TTL.",
    "Integration tests use pytest fixtures for database setup.",
]
query = "version control for database tables"

for model_name in (MODEL_SMALL, MODEL_LARGE):
    c_vecs, c_tok = embed(model_name, corpus)
    q_vecs, q_tok = embed(model_name, [query])
    scores = [cosine(q_vecs[0], v) for v in c_vecs]
    best = int(np.argmax(scores))
    print(f"\n{model_name}  (tokens: {c_tok + q_tok})")
    print(f"  best=[{best}] score={scores[best]:.4f}  {corpus[best][:55]}...")

---

## 15. Demonstration: Preserve Input Order

Always sort by `index` when mapping vectors back to source records.

In [ ]:
ids = ["chunk_a", "chunk_b", "chunk_c"]
texts = [f"Document text for {i}" for i in ids]

response = client.embeddings.create(model=MODEL_SMALL, input=texts)
id_by_index = {i: chunk_id for i, chunk_id in enumerate(ids)}

print("API returned order (may differ from input order):")
for item in response.data:
    mapped_id = id_by_index[item.index]
    print(f"  response position → input index={item.index} → id={mapped_id}")

---

## 16. Environment and Configuration

Production apps externalize model choice:

```bash
# .env
OPENAI_API_KEY=sk-...
EMBEDDING_MODEL=text-embedding-3-small
EMBEDDING_DIMENSIONS=1536
```

```python
import os
MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
DIM = int(os.getenv("EMBEDDING_DIMENSIONS", "1536"))
```

Log `model` and `dimensions` on every ingest and search (notebook 10).

---

## 17. Errors and Retries

| HTTP / error | Action |
|--------------|--------|
| **401** | Invalid `OPENAI_API_KEY` |
| **429** | Rate limit — backoff, smaller batches |
| **400** | Text too long, bad model name, invalid `dimensions` |
| **500** | Retry with exponential backoff |

Ingest jobs should be **resumable** — checkpoint which chunk IDs were embedded.

---

## 18. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Chat model for embeddings | Wrong vectors | Use embeddings endpoint |
| Different dims index vs query | Broken similarity | Same model + dimensions |
| One string per request at scale | Slow, costly round trips | Batch |
| Embed full PDF once | Vague retrieval | Chunk first (nb 04) |
| Switch model without re-index | Garbage rankings | New collection + re-embed |
| Ignore `item.index` | Shuffled metadata | Sort by index |

---

## 19. Summary

Use the **Embeddings API** (not chat) to obtain vectors. Start with **`text-embedding-3-small`**; upgrade to **`3-large`** only when evals justify cost. **Batch** inputs, **count tokens** with tiktoken, and keep **`dimensions`** consistent across index and query.

Choose providers based on deployment, privacy, scale, and eval results — OpenAI for this curriculum, Gemini for Google stacks, sentence-transformers for offline. Cache embeddings and version model settings in config.

Next: **04. Document Chunking for Retrieval** — split documents before embedding.